# SQLi Detection Model — CNN + Bi-LSTM (NumPy from scratch)

This notebook trains the SQL Injection detection model described in the project proposal.

## Architecture
```
Input (token IDs, seq_len=256)
  → Embedding  (vocab_size × 64)
  → Conv1D + ReLU + GlobalMaxPool  (64 filters, kernel=3)
  → Bi-LSTM  (hidden=32 each direction → 64 combined)
  → Dense + ReLU  (128 → 64)
  → Dense + Sigmoid  (64 → 1)  →  risk score [0, 1]
```

## Semantic signal tokens (vocabulary = 173)
The preprocessing pipeline injects four special tokens the model learns to recognise:

| Token | Meaning |
|---|---|
| `FSTRING_SQL` | f-string with SQL content — variables interpolated directly |
| `UNSAFE_EXEC` | `execute(query)` with no parameter tuple — dangerous |
| `SAFE_EXEC` | `execute(query, (params,))` — parameterized, safe |
| `SQL_CONCAT` | `SQL_STRING + variable` — concatenation injection |

## After training
Download `sqli_model.npz` and place it in:
```
backend/app/model/weights/sqli_model.npz
```
Then restart the backend — the model loads automatically.

## Required files
Upload before running:
- `vocabulary.json` (from `backend/colab_export/`)
- `training_data.npz` (from `backend/colab_export/`)


## 1. Upload files and load data

Upload `vocabulary.json` and `training_data.npz` from `backend/colab_export/`.

**Option A** — Upload via Files panel (left sidebar → upload icon)

**Option B** — Mount Google Drive and copy:
```python
from google.colab import drive
drive.mount('/content/drive')
# Then copy files from your Drive path
```


In [ ]:
from google.colab import files
import json, numpy as np, os

# ── Load vocabulary ───────────────────────────────────────────────────────
with open("vocabulary.json") as f:
    vocab = json.load(f)
VOCAB_SIZE = len(vocab)
print(f"Vocabulary size: {VOCAB_SIZE}")

# ── Load dataset ──────────────────────────────────────────────────────────
data = np.load("training_data.npz")
X = data["X"]   # (N, 256) int32 — token ID sequences
y = data["y"]   # (N,)    float32 — 1.0=vulnerable, 0.0=safe
n_vuln = int(y.sum())
n_safe = int((1 - y).sum())
print(f"Dataset: {len(X)} samples ({n_vuln} vulnerable, {n_safe} safe)")
print(f"Sequence length: {X.shape[1]}")

# ── Show semantic signal coverage ─────────────────────────────────────────
signal_ids = {
    'FSTRING_SQL': vocab.get('FSTRING_SQL', -1),
    'UNSAFE_EXEC': vocab.get('UNSAFE_EXEC', -1),
    'SAFE_EXEC':   vocab.get('SAFE_EXEC',   -1),
    'SQL_CONCAT':  vocab.get('SQL_CONCAT',  -1),
}
print()
print("Semantic signal coverage in dataset:")
X_vuln = X[y == 1]
X_safe  = X[y == 0]
for name, sid in signal_ids.items():
    if sid < 0:
        print(f"  {name}: NOT IN VOCABULARY — re-run export_for_colab.py")
        continue
    if name in ('FSTRING_SQL', 'UNSAFE_EXEC', 'SQL_CONCAT'):
        count = int(np.any(X_vuln == sid, axis=1).sum())
        pct   = 100 * count // len(X_vuln)
        print(f"  {name:15s}: {count:3d}/{len(X_vuln)} vulnerable samples ({pct}%)")
    else:  # SAFE_EXEC
        count = int(np.any(X_safe == sid, axis=1).sum())
        pct   = 100 * count // len(X_safe)
        print(f"  {name:15s}: {count:3d}/{len(X_safe)} safe samples ({pct}%)")


## 2. Hyper-parameters

These must match  exactly.

In [ ]:
# ── Architecture constants (MUST match backend/app/model/sqli_detector.py) ──
EMBED_DIM    = 64
CONV_FILTERS = 64
KERNEL_SIZE  = 3
LSTM_HIDDEN  = 32   # per direction; BiLSTM output = 2*32 = 64
DENSE_HIDDEN = 64
DENSE_IN     = CONV_FILTERS + 2 * LSTM_HIDDEN   # = 128
MODEL_SEQ_LEN = 256

# ── Training hyper-parameters ────────────────────────────────────────────
EPOCHS   = 40
LR_INIT  = 0.008
LR_DECAY = 0.94       # multiply lr by this each epoch
SEED     = 42

print("Config OK")

## 3. Activation helpers

In [ ]:
import numpy as np

def sigmoid(x):
    return np.where(x >= 0, 1/(1+np.exp(-x)), np.exp(x)/(1+np.exp(x)))

def sigmoid_grad(s): return s * (1 - s)

def relu(x): return np.maximum(0.0, x)

def relu_grad(x): return (x > 0).astype(np.float32)

def tanh_grad(t): return 1.0 - t**2


## 4. Weight initialisation

In [ ]:
rng = np.random.default_rng(SEED)

def he(shape):
    """He (Kaiming) initialisation for ReLU layers."""
    fan_in = shape[-1] if len(shape) > 1 else shape[0]
    return rng.normal(0, np.sqrt(2.0/fan_in), shape).astype(np.float32)

# Embedding
emb_W = rng.normal(0, 0.05, (VOCAB_SIZE, EMBED_DIM)).astype(np.float32)

# Conv1D  (CONV_FILTERS, EMBED_DIM, KERNEL_SIZE)
conv_W = he((CONV_FILTERS, EMBED_DIM, KERNEL_SIZE))
conv_b = np.zeros(CONV_FILTERS, dtype=np.float32)

# BiLSTM — each direction: W shape = (4*H, EMBED_DIM + H)
bilstm_fwd_W = he((4*LSTM_HIDDEN, EMBED_DIM + LSTM_HIDDEN))
bilstm_fwd_b = np.zeros(4*LSTM_HIDDEN, dtype=np.float32)
bilstm_bwd_W = he((4*LSTM_HIDDEN, EMBED_DIM + LSTM_HIDDEN))
bilstm_bwd_b = np.zeros(4*LSTM_HIDDEN, dtype=np.float32)

# Dense layers
dense1_W = he((DENSE_HIDDEN, DENSE_IN))
dense1_b = np.zeros(DENSE_HIDDEN, dtype=np.float32)
dense2_W = he((1, DENSE_HIDDEN))
dense2_b = np.zeros(1, dtype=np.float32)

print("Weights initialised")
print(f"  emb_W:         {emb_W.shape}")
print(f"  conv_W:        {conv_W.shape}")
print(f"  bilstm_fwd_W:  {bilstm_fwd_W.shape}")
print(f"  dense1_W:      {dense1_W.shape}")
print(f"  dense2_W:      {dense2_W.shape}")

## 5. Forward pass helpers

In [ ]:
def fwd_embedding(ids):
    return emb_W[ids]   # (seq_len, EMBED_DIM)


def fwd_conv1d_maxpool(emb):
    seq_len = emb.shape[0]
    out_len = max(1, seq_len - KERNEL_SIZE + 1)
    W_flat = conv_W.reshape(CONV_FILTERS, -1)   # (F, k*D)
    conv_out = np.zeros((out_len, CONV_FILTERS), dtype=np.float32)
    for pos in range(out_len):
        patch = emb[pos:pos+KERNEL_SIZE].flatten()
        conv_out[pos] = W_flat @ patch + conv_b
    act = relu(conv_out)
    pool_idx = np.argmax(act, axis=0)
    pooled = act[pool_idx, np.arange(CONV_FILTERS)]
    return pooled, act, pool_idx, conv_out   # for backprop


def lstm_step(x_t, h, c, W, b):
    H = LSTM_HIDDEN
    concat = np.concatenate([x_t, h])
    gates = W @ concat + b
    f = sigmoid(gates[0*H:1*H])
    i = sigmoid(gates[1*H:2*H])
    o = sigmoid(gates[2*H:3*H])
    g = np.tanh(gates[3*H:4*H])
    c_new = f * c + i * g
    tanh_c = np.tanh(c_new)
    h_new = o * tanh_c
    return h_new, c_new, (x_t, h, c, f, i, o, g, c_new, tanh_c, concat)


def fwd_lstm_direction(emb, W, b, reverse=False):
    H = LSTM_HIDDEN
    seq_len = emb.shape[0]
    h = np.zeros(H, dtype=np.float32)
    c = np.zeros(H, dtype=np.float32)
    caches = []
    order = range(seq_len) if not reverse else reversed(range(seq_len))
    for t in order:
        h, c, cache = lstm_step(emb[t], h, c, W, b)
        caches.append(cache)
    if reverse:
        caches.reverse()
    return h, caches


def fwd_dense(x, W, b, act):
    pre = W @ x + b
    out = relu(pre) if act == "relu" else sigmoid(pre)
    return out, pre


def forward(ids):
    emb = fwd_embedding(ids)
    cnn_out, cnn_act, pool_idx, conv_pre = fwd_conv1d_maxpool(emb)
    h_fwd, fwd_caches = fwd_lstm_direction(emb, bilstm_fwd_W, bilstm_fwd_b, reverse=False)
    h_bwd, bwd_caches = fwd_lstm_direction(emb, bilstm_bwd_W, bilstm_bwd_b, reverse=True)
    lstm_out = np.concatenate([h_fwd, h_bwd])
    combined = np.concatenate([cnn_out, lstm_out])
    h1, pre1 = fwd_dense(combined, dense1_W, dense1_b, "relu")
    score, pre2 = fwd_dense(h1, dense2_W, dense2_b, "sigmoid")
    cache = (emb, ids, cnn_act, pool_idx, conv_pre,
             fwd_caches, bwd_caches, combined, h1, pre1, score, pre2)
    return float(score[0]), cache

print("Forward pass helpers defined")

## 6. Backward pass (BPTT) helpers

In [ ]:
CLIP = 5.0

def clip(g): return np.clip(g, -CLIP, CLIP)


def bwd_lstm_direction(caches, d_h_last, W, b, reversed_iter=True):
    """
    BPTT for one LSTM direction.

    For the FORWARD LSTM: computation went t=0..T-1, so BPTT goes T-1..0.
    Use reversed_iter=True (default).

    For the BACKWARD LSTM: computation went t=T-1..0, caches were then
    reversed so caches[i] = t=i. BPTT must go t=0..T-1 (forward order).
    Use reversed_iter=False.

    Returns (d_x_seq, dW, db) where d_x_seq[i] = gradient at position i.
    """
    H = LSTM_HIDDEN
    dW = np.zeros_like(W)
    db = np.zeros_like(b)
    d_h = d_h_last.copy()
    d_c = np.zeros(H, dtype=np.float32)
    seq_len = len(caches)
    d_x_seq = [None] * seq_len

    order = reversed(range(seq_len)) if reversed_iter else range(seq_len)

    for t in order:
        x_t, h_prev, c_prev, f, i_g, o, g, c_t, tanh_c, concat = caches[t]
        d_o  = d_h * tanh_c
        d_tc = d_h * o
        d_ct = d_c + d_tc * tanh_grad(tanh_c)
        d_f  = d_ct * c_prev
        d_i  = d_ct * g
        d_g  = d_ct * i_g
        d_c  = d_ct * f
        gates = np.concatenate([
            d_f * sigmoid_grad(f),
            d_i * sigmoid_grad(i_g),
            d_o * sigmoid_grad(o),
            d_g * tanh_grad(g),
        ])
        dW += np.outer(gates, concat)
        db += gates
        d_concat = W.T @ gates
        x_dim = x_t.shape[0]
        d_x_seq[t] = d_concat[:x_dim]
        d_h = d_concat[x_dim:]

    return d_x_seq, dW, db


def train_step(ids, label, lr):
    global emb_W, conv_W, conv_b
    global bilstm_fwd_W, bilstm_fwd_b, bilstm_bwd_W, bilstm_bwd_b
    global dense1_W, dense1_b, dense2_W, dense2_b

    # ── Forward ──────────────────────────────────────────────────────
    emb = fwd_embedding(ids)
    cnn_out, cnn_act, pool_idx, conv_pre = fwd_conv1d_maxpool(emb)
    h_fwd, fwd_caches = fwd_lstm_direction(emb, bilstm_fwd_W, bilstm_fwd_b, reverse=False)
    h_bwd, bwd_caches = fwd_lstm_direction(emb, bilstm_bwd_W, bilstm_bwd_b, reverse=True)
    lstm_out = np.concatenate([h_fwd, h_bwd])
    combined = np.concatenate([cnn_out, lstm_out])
    h1, pre1 = fwd_dense(combined, dense1_W, dense1_b, "relu")
    score, pre2 = fwd_dense(h1, dense2_W, dense2_b, "sigmoid")
    pred = float(score[0])

    # ── Loss (Binary Cross-Entropy) ───────────────────────────────────
    eps = 1e-7
    p = np.clip(pred, eps, 1 - eps)
    loss = -(label * np.log(p) + (1 - label) * np.log(1 - p))

    # ── Backward through Dense 2 ──────────────────────────────────────
    d_score = np.array([pred - label], dtype=np.float32)   # dBCE/doutput
    d_h1 = clip(dense2_W.T @ d_score)
    dense2_W -= lr * clip(np.outer(d_score, h1))
    dense2_b -= lr * clip(d_score)

    # ── Backward through Dense 1 ──────────────────────────────────────
    d_pre1 = d_h1 * relu_grad(pre1)
    d_combined = clip(dense1_W.T @ d_pre1)
    dense1_W -= lr * clip(np.outer(d_pre1, combined))
    dense1_b -= lr * clip(d_pre1)

    # ── Split gradient for CNN and BiLSTM ─────────────────────────────
    d_cnn   = d_combined[:CONV_FILTERS]
    d_lstm  = d_combined[CONV_FILTERS:]
    d_h_fwd = d_lstm[:LSTM_HIDDEN]
    d_h_bwd = d_lstm[LSTM_HIDDEN:]

    # Fwd LSTM: last computation at t=T-1 → reversed_iter=True (default)
    d_x_fwd, dW_fwd, db_fwd = bwd_lstm_direction(
        fwd_caches, d_h_fwd, bilstm_fwd_W, bilstm_fwd_b, reversed_iter=True
    )
    # Bwd LSTM: last computation at t=0 → reversed_iter=False
    d_x_bwd, dW_bwd, db_bwd = bwd_lstm_direction(
        bwd_caches, d_h_bwd, bilstm_bwd_W, bilstm_bwd_b, reversed_iter=False
    )
    bilstm_fwd_W -= lr * clip(dW_fwd)
    bilstm_fwd_b -= lr * clip(db_fwd)
    bilstm_bwd_W -= lr * clip(dW_bwd)
    bilstm_bwd_b -= lr * clip(db_bwd)

    # ── Backward through Conv1D + MaxPool ─────────────────────────────
    seq_len = emb.shape[0]
    out_len = max(1, seq_len - KERNEL_SIZE + 1)
    d_act = np.zeros((out_len, CONV_FILTERS), dtype=np.float32)
    d_act[pool_idx, np.arange(CONV_FILTERS)] = d_cnn
    d_conv = d_act * relu_grad(conv_pre)
    W_flat = conv_W.reshape(CONV_FILTERS, -1)
    d_emb_cnn = np.zeros_like(emb)
    dW_conv = np.zeros_like(conv_W.reshape(CONV_FILTERS, -1))
    for pos in range(out_len):
        patch = emb[pos:pos+KERNEL_SIZE].flatten()
        dW_conv += np.outer(d_conv[pos], patch)
        d_patch = W_flat.T @ d_conv[pos]
        d_emb_cnn[pos:pos+KERNEL_SIZE] += d_patch.reshape(KERNEL_SIZE, -1)
    conv_W -= lr * clip(dW_conv).reshape(conv_W.shape)
    conv_b -= lr * clip(d_conv.sum(axis=0))

    # ── Backward through Embedding ────────────────────────────────────
    d_emb_lstm = np.zeros_like(emb)
    for t in range(seq_len):
        if d_x_fwd[t] is not None: d_emb_lstm[t] += d_x_fwd[t]
        if d_x_bwd[t] is not None: d_emb_lstm[t] += d_x_bwd[t]
    d_emb = clip(d_emb_cnn + d_emb_lstm)
    np.add.at(emb_W, ids, -lr * d_emb)

    return loss


print("Backward pass helpers defined (Bi-LSTM BPTT corrected)")


## 7. Training loop

In [ ]:
import time

rng_train = np.random.default_rng(SEED)
n = len(X)
split = int(n * 0.80)
idx_all = rng_train.permutation(n)
X_train, y_train = X[idx_all[:split]], y[idx_all[:split]]
X_val,   y_val   = X[idx_all[split:]], y[idx_all[split:]]
print(f"Train: {len(X_train)}   Val: {len(X_val)}")


# ── Evaluation metrics (Precision, Recall, F1) ────────────────────────────

def evaluate(Xs, ys, threshold=0.5):
    """Compute accuracy, precision, recall, F1, and confusion matrix."""
    preds = np.array([forward(Xs[i])[0] for i in range(len(Xs))])
    binary = (preds >= threshold).astype(int)
    labels = ys.astype(int)

    tp = int(np.sum((binary == 1) & (labels == 1)))
    tn = int(np.sum((binary == 0) & (labels == 0)))
    fp = int(np.sum((binary == 1) & (labels == 0)))
    fn = int(np.sum((binary == 0) & (labels == 1)))

    acc  = (tp + tn) / len(labels)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

    return {
        "acc": round(acc,  4),
        "prec": round(prec, 4),
        "rec":  round(rec,  4),
        "f1":   round(f1,   4),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }


# ── Training loop ─────────────────────────────────────────────────────────

PATIENCE = 8            # early stopping patience (epochs without improvement)

lr = LR_INIT
best_val_f1  = 0.0
no_improve   = 0
history = []

for epoch in range(1, EPOCHS + 1):
    perm = rng_train.permutation(len(X_train))
    total_loss = 0.0
    t0 = time.time()

    for i in perm:
        loss = train_step(X_train[i], float(y_train[i]), lr)
        total_loss += loss

    avg_loss = total_loss / len(perm)
    val_m  = evaluate(X_val,   y_val)
    tr_m   = evaluate(X_train, y_train)
    elapsed = time.time() - t0

    history.append({
        "epoch": epoch, "loss": avg_loss,
        "val_f1": val_m["f1"], "val_acc": val_m["acc"],
        "val_prec": val_m["prec"], "val_rec": val_m["rec"],
    })

    print(
        f"Epoch {epoch:02d}/{EPOCHS}  "
        f"loss={avg_loss:.4f}  "
        f"tr_acc={tr_m['acc']:.3f}  tr_f1={tr_m['f1']:.3f}  "
        f"val_acc={val_m['acc']:.3f}  val_f1={val_m['f1']:.3f}  "
        f"P={val_m['prec']:.3f}  R={val_m['rec']:.3f}  "
        f"lr={lr:.5f}  ({elapsed:.1f}s)"
    )

    # Save best model (tracked by val F1, not accuracy)
    if val_m["f1"] > best_val_f1:
        best_val_f1 = val_m["f1"]
        no_improve = 0
        np.savez("sqli_model.npz",
                 emb_W=emb_W,
                 conv_W=conv_W, conv_b=conv_b,
                 bilstm_fwd_W=bilstm_fwd_W, bilstm_fwd_b=bilstm_fwd_b,
                 bilstm_bwd_W=bilstm_bwd_W, bilstm_bwd_b=bilstm_bwd_b,
                 dense1_W=dense1_W, dense1_b=dense1_b,
                 dense2_W=dense2_W, dense2_b=dense2_b)
        print(f"   ✓ Best model saved (val_f1={val_m['f1']:.3f})")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"   ⏹ Early stopping at epoch {epoch} (no F1 improvement for {PATIENCE} epochs)")
            break

    lr *= LR_DECAY

print(f"\nTraining complete. Best val_F1 = {best_val_f1:.4f}")


## 8a. Final evaluation report

Loads the **best checkpoint** and computes Precision, Recall, F1, and Confusion Matrix
on the held-out validation set.


In [ ]:
# ── Final evaluation on validation set (using best saved weights) ────────
# Reload best weights
best = np.load("sqli_model.npz")
emb_W        = best["emb_W"]
conv_W       = best["conv_W"];  conv_b        = best["conv_b"]
bilstm_fwd_W = best["bilstm_fwd_W"]; bilstm_fwd_b = best["bilstm_fwd_b"]
bilstm_bwd_W = best["bilstm_bwd_W"]; bilstm_bwd_b = best["bilstm_bwd_b"]
dense1_W     = best["dense1_W"]; dense1_b      = best["dense1_b"]
dense2_W     = best["dense2_W"]; dense2_b      = best["dense2_b"]

val_metrics = evaluate(X_val, y_val)

print("=" * 52)
print("FINAL MODEL EVALUATION (Validation Set)")
print("=" * 52)
print(f"  Accuracy:  {val_metrics['acc']:.4f}")
print(f"  Precision: {val_metrics['prec']:.4f}")
print(f"  Recall:    {val_metrics['rec']:.4f}")
print(f"  F1 Score:  {val_metrics['f1']:.4f}")
print()
print("Confusion Matrix:")
print("                  Pred SAFE   Pred VULN")
print(f"  Actual SAFE       {val_metrics['tn']:>6}      {val_metrics['fp']:>6}")
print(f"  Actual VULN       {val_metrics['fn']:>6}      {val_metrics['tp']:>6}")
print()
tp = val_metrics["tp"]; tn = val_metrics["tn"]
fp = val_metrics["fp"]; fn = val_metrics["fn"]
print(f"  True  Positives (correctly flagged vulnerable): {tp}")
print(f"  True  Negatives (correctly flagged safe):       {tn}")
print(f"  False Positives (safe flagged as vulnerable):   {fp}")
print(f"  False Negatives (vulnerable missed!):           {fn}  <-- most critical")
print("=" * 52)


## 8b. Training history (optional plot)


In [ ]:
# Training history — uncomment to plot if matplotlib is available
try:
    import matplotlib.pyplot as plt
    epochs_x = [h["epoch"] for h in history]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(epochs_x, [h["loss"] for h in history], label="Train Loss")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Training Loss")
    ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(epochs_x, [h["val_f1"]  for h in history], label="Val F1",  color="green")
    ax2.plot(epochs_x, [h["val_acc"] for h in history], label="Val Acc", color="blue", linestyle="--")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Score"); ax2.set_title("Validation Metrics")
    ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("training_history.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Plot saved as training_history.png")
except Exception as e:
    print(f"Plot skipped: {e}")


## 9. Verification — test the saved model

Simulates exactly what `backend/app/model/sqli_detector.py` does.
Runs a forward pass with the best saved weights to confirm they load correctly.


In [ ]:
# Load the saved weights and run forward pass to verify
w = np.load("sqli_model.npz")

def verify_forward(ids_list):
    ids = np.array(ids_list[:MODEL_SEQ_LEN], dtype=np.int32)
    if len(ids) < MODEL_SEQ_LEN:
        ids = np.concatenate([ids, np.zeros(MODEL_SEQ_LEN-len(ids), dtype=np.int32)])
    emb = w["emb_W"][ids]
    # Conv
    out_len = max(1, MODEL_SEQ_LEN - KERNEL_SIZE + 1)
    W_flat = w["conv_W"].reshape(CONV_FILTERS, -1)
    conv_out = np.array([W_flat @ emb[p:p+KERNEL_SIZE].flatten() + w["conv_b"] for p in range(out_len)])
    cnn_out = np.max(np.maximum(0, conv_out), axis=0)
    # BiLSTM
    def run_dir(W, b, rev):
        H = LSTM_HIDDEN
        h = np.zeros(H); c = np.zeros(H)
        order = range(MODEL_SEQ_LEN) if not rev else reversed(range(MODEL_SEQ_LEN))
        for t in order:
            cat = np.concatenate([emb[t], h])
            gs = W @ cat + b
            f=sigmoid(gs[:H]); ig=sigmoid(gs[H:2*H]); o=sigmoid(gs[2*H:3*H]); g=np.tanh(gs[3*H:])
            c=f*c+ig*g; h=o*np.tanh(c)
        return h
    lstm_out = np.concatenate([run_dir(w["bilstm_fwd_W"],w["bilstm_fwd_b"],False),
                                run_dir(w["bilstm_bwd_W"],w["bilstm_bwd_b"],True)])
    combined = np.concatenate([cnn_out, lstm_out])
    h1 = np.maximum(0, w["dense1_W"] @ combined + w["dense1_b"])
    score = sigmoid(w["dense2_W"] @ h1 + w["dense2_b"])
    return float(score[0])

# Spot-check a vulnerable sample and a safe sample
PAD_ID = vocab["PAD"]
UNK_ID = vocab["UNK"]
SQL_ID = vocab.get("SQL_STRING", UNK_ID)
VAR0   = vocab.get("VAR_0", UNK_ID)
PLUS   = vocab.get("+", UNK_ID)
vuln_ids = [VAR0, 61, SQL_ID, PLUS, VAR0] + [PAD_ID]*(MODEL_SEQ_LEN-5)
safe_ids = [PAD_ID]*MODEL_SEQ_LEN
print(f"Vulnerable-like score: {verify_forward(vuln_ids):.4f}  (expect > 0.5)")
print(f"PAD-only score:        {verify_forward(safe_ids):.4f}  (expect < 0.5)")

---
# Model 2 — Fix Recommendation

Model 2 is a **separate classification model** from Model 1.

**Responsibilities:**
- Receives: the vulnerable code context + vulnerability type from Model 1
- Produces: which fix strategy to apply (A/B/C/D)
- Does NOT generate code itself — the backend templates do that from the predicted class

**Fix classes:**
| Class | Strategy | When |
|---|---|---|
| A | Parameterized Query | f-string / concat / format injection |
| B | Whitelist Validation | dynamic column/table names |
| C | ORM migration | complex multi-query patterns |
| D | Second-Order | value stored then re-used unsafely |

**Architecture:** Embedding → GlobalAvgPool → Dense(ReLU) → Dense(Softmax)

> Model 2 is simpler than Model 1 because its task is classification among
> 4 fix strategies, not sequence modeling across hundreds of tokens.


In [ ]:
# ── Model 2 Dataset ──────────────────────────────────────────────────────
# Re-use the same token sequences from training_data.npz.
# For vulnerable samples, assign fix class based on signals present.

# Fix class mapping:
#   0 = A (Parameterized Query) — FSTRING_SQL, SQL_CONCAT, UNSAFE_EXEC
#   1 = B (Whitelist Validation) — ORDER BY injection, dynamic table
#   2 = C (ORM)  — reserved for future use
#   3 = D (Second-Order) — reserved for future use
#
# For this dataset almost all vulnerabilities are class A.

FSTRING_ID  = vocab.get('FSTRING_SQL',  -1)
SQL_CONC_ID = vocab.get('SQL_CONCAT',   -1)
UNSAFE_ID   = vocab.get('UNSAFE_EXEC',  -1)

# Select only vulnerable samples
X_vuln = X[y == 1]

def assign_fix_class(seq):
    ids_set = set(seq.tolist())
    # Class B: ORDER BY patterns (no strong signal in current vocab — default A)
    # Class A: any injection via FSTRING, CONCAT, or UNSAFE_EXEC
    if FSTRING_ID in ids_set or SQL_CONC_ID in ids_set or UNSAFE_ID in ids_set:
        return 0  # Fix A
    return 0  # Default to Fix A for all current samples

y2 = np.array([assign_fix_class(seq) for seq in X_vuln], dtype=np.int32)
print(f'Model 2 dataset: {len(X_vuln)} vulnerable samples')
print(f'Fix class distribution: {dict(zip(*np.unique(y2, return_counts=True)))}')


In [ ]:
# ── Model 2 Weight Initialisation ────────────────────────────────────────
# Architecture: Embedding (shared) → GlobalAvgPool → Dense(ReLU) → Dense(Softmax)
# Much simpler than Model 1 — no CNN or BiLSTM needed.

NUM_CLASSES_M2 = 4   # A, B, C, D
M2_HIDDEN = 32

# GlobalAvgPool output size = EMBED_DIM
# Model 2 shares the same embedding as Model 1 (frozen during M2 training)
m2_dense1_W = he((M2_HIDDEN, EMBED_DIM))
m2_dense1_b = np.zeros(M2_HIDDEN, dtype=np.float32)
m2_dense2_W = he((NUM_CLASSES_M2, M2_HIDDEN))
m2_dense2_b = np.zeros(NUM_CLASSES_M2, dtype=np.float32)

print(f'm2_dense1_W: {m2_dense1_W.shape}  ({EMBED_DIM} → {M2_HIDDEN})')
print(f'm2_dense2_W: {m2_dense2_W.shape}  ({M2_HIDDEN} → {NUM_CLASSES_M2} classes)')


In [ ]:
# ── Model 2 Forward Pass ─────────────────────────────────────────────────

def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

def m2_forward(ids):
    """Embedding → GlobalAvgPool → Dense(ReLU) → Dense(Softmax)"""
    emb = emb_W[ids]                          # (seq_len, EMBED_DIM)
    pooled = emb.mean(axis=0)                 # (EMBED_DIM,) global avg pool
    h = relu(m2_dense1_W @ pooled + m2_dense1_b)  # (M2_HIDDEN,)
    logits = m2_dense2_W @ h + m2_dense2_b   # (NUM_CLASSES_M2,)
    probs = softmax(logits)
    return probs

# Test
test_probs = m2_forward(X_vuln[0])
print(f'Model 2 test output (probabilities over 4 fix classes): {test_probs.round(3)}')
print(f'Predicted fix class: {np.argmax(test_probs)} (A=0, B=1, C=2, D=3)')


In [ ]:
# ── Model 2 Training ─────────────────────────────────────────────────────
# Categorical cross-entropy, SGD, no BPTT needed.

M2_EPOCHS = 20
M2_LR     = 0.01

def m2_train_step(ids, label_idx, lr):
    global m2_dense1_W, m2_dense1_b, m2_dense2_W, m2_dense2_b

    # Forward
    emb    = emb_W[ids]
    pooled = emb.mean(axis=0)
    pre1   = m2_dense1_W @ pooled + m2_dense1_b
    h      = relu(pre1)
    logits = m2_dense2_W @ h + m2_dense2_b
    probs  = softmax(logits)

    # Categorical cross-entropy loss
    loss = -np.log(np.clip(probs[label_idx], 1e-7, 1.0))

    # Backward through Dense 2 (Softmax + CCE gradient)
    d_logits = probs.copy()
    d_logits[label_idx] -= 1.0
    m2_dense2_W -= lr * np.outer(d_logits, h)
    m2_dense2_b -= lr * d_logits
    d_h = m2_dense2_W.T @ d_logits

    # Backward through Dense 1 (ReLU)
    d_pre1 = d_h * relu_grad(pre1)
    m2_dense1_W -= lr * np.outer(d_pre1, pooled)
    m2_dense1_b -= lr * d_pre1
    # (Embedding not updated during M2 training — it belongs to Model 1)

    return float(loss)

rng_m2 = np.random.default_rng(99)
best_m2_acc = 0.0

for epoch in range(1, M2_EPOCHS + 1):
    perm = rng_m2.permutation(len(X_vuln))
    total = sum(m2_train_step(X_vuln[i], int(y2[i]), M2_LR) for i in perm)
    preds = np.array([np.argmax(m2_forward(x)) for x in X_vuln])
    acc = float((preds == y2).mean())
    if acc > best_m2_acc:
        best_m2_acc = acc
        np.savez('sqli_fix_model.npz',
                 m2_dense1_W=m2_dense1_W, m2_dense1_b=m2_dense1_b,
                 m2_dense2_W=m2_dense2_W, m2_dense2_b=m2_dense2_b)
    print(f'M2 Epoch {epoch:02d}/{M2_EPOCHS}  loss={total/len(X_vuln):.4f}  acc={acc:.3f}')

print(f'\nModel 2 training complete. Best accuracy: {best_m2_acc:.3f}')
print('Note: high accuracy expected because most samples are class A.')


In [ ]:
# ── Download Model 2 weights ─────────────────────────────────────────────
from google.colab import files

files.download('sqli_fix_model.npz')
print('Downloaded sqli_fix_model.npz')
print('Place it in: backend/app/model/weights/sqli_fix_model.npz')
print()
print('Also download Model 1 weights if not done yet:')
files.download('sqli_model.npz')
print('Place it in: backend/app/model/weights/sqli_model.npz')


## 10. Download weights

Download `sqli_model.npz` and place it in `backend/app/model/weights/sqli_model.npz`.
Then restart the backend — the model loads automatically on startup.


In [ ]:
from google.colab import files
files.download("sqli_model.npz")
print("Downloaded sqli_model.npz")
print("Place it in:  backend/app/model/weights/sqli_model.npz")